# FilaNet: end-to-end reproduction

Solar Filament Segmentation Challenge 2026. This notebook rebuilds the submission from the stored weights
(no training): **test images -> physics-aware inputs -> FilaNet (4.5 M params) -> mask + spine maps ->
hysteresis components -> keep rule / rescorer -> `submission.csv`**.

Runs on CPU (tiled) or GPU (one full-disk pass per image). Every step calls the same functions as
`scripts/predict.py`.

In [ ]:
import os, sys, glob, json, time, hashlib
import numpy as np, torch, matplotlib.pyplot as plt
REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, REPO)
from filanet.data import load_image
from filanet.geometry import make_inputs, longitude_ignore_mask
from filanet.infer import predict_maps
from filanet.model import load_checkpoint, count_params
from filanet.metric import encode
from filanet.postproc import label_components, component_features, rule_keep
device = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_DIR = os.environ.get('TEST_DIR') or sorted(glob.glob('/kaggle/input/**/test_images', recursive=True) + glob.glob(os.path.join(REPO, 'data', '**', 'test_images'), recursive=True))[0]
print('device', device, '| test images', TEST_DIR)

## 1. Model and post-processing configuration

In [ ]:
model, ck = load_checkpoint(os.path.join(REPO, 'weights', 'filanet.pt'), device)
post = json.load(open(os.path.join(REPO, 'weights', 'postproc.json')))
rescorer = None
if post.get('kind') == 'rescorer':
    from filanet.rescore import Rescorer
    rescorer = Rescorer(os.path.join(REPO, 'weights', 'rescorer.txt'))
print(f"FilaNet {ck['cfg']['encoder']} | {count_params(model)/1e6:.2f} M params | post-processing: {post}")

## 2. Inputs: the three physics-aware channels

Disk-normalised intensity, local contrast (limb darkening removed) and radial position r/R. The hatched
band is beyond 70 deg longitude, where MAGFiLO annotators were told not to annotate.

In [ ]:
stems = sorted(os.path.splitext(f)[0] for f in os.listdir(TEST_DIR))
exts = {os.path.splitext(f)[0]: f for f in os.listdir(TEST_DIR)}
img0 = load_image(os.path.join(TEST_DIR, exts[stems[0]]))
x0, disk0 = make_inputs(img0)
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(img0, cmap='gray'); ax[0].set_title(stems[0])
for i, t in enumerate(['normalised intensity', 'local contrast', 'r / R']):
    ax[i + 1].imshow(x0[i], cmap='gray'); ax[i + 1].set_title(t)
ax[0].contour(longitude_ignore_mask(img0.shape, disk0), [0.5], colors='orange')
for a in ax: a.axis('off')
plt.show()

## 3. Predict every test image and extract instances

In [ ]:
def extract(pm, sm, disk, img):
    lab, n = label_components(pm, post['t_lo'], post['t_hi'], post.get('close_px', 0))
    feats = component_features(lab, n, pm, sm, disk, make_inputs(img, disk)[0][1])
    keep = rescorer.keep(feats) if rescorer else rule_keep(feats, post.get('min_area', 100), post.get('min_meanp', 0.0), post.get('max_sin_lon'))
    return [(lab == i + 1) for i in np.flatnonzero(keep)]

rows, gallery, t0 = [], {}, time.time()
for n, stem in enumerate(stems):
    img = load_image(os.path.join(TEST_DIR, exts[stem]))
    pm, sm, disk = predict_maps(model, img, device=device, tile=None if device == 'cuda' else 1024)
    masks = extract(pm, sm, disk, img)
    rows += [(f'{stem}_{k}', encode(m)['counts'].decode('ascii')) for k, m in enumerate(masks, 1)]
    if n < 4: gallery[stem] = (img, pm, sm, masks)
print(f'{len(rows)} instances on {len({r[0].rsplit("_",1)[0] for r in rows})}/{len(stems)} images in {time.time()-t0:.0f}s')

## 4. Write the submission

In [ ]:
import csv
with open('submission.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['filament_id', 'segmentation_rle']); w.writerows(rows)
print('md5', hashlib.md5(open('submission.csv', 'rb').read()).hexdigest())

## 5. Gallery: probability, spine and final instances

In [ ]:
for stem, (img, pm, sm, masks) in gallery.items():
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(img, cmap='gray'); ax[0].set_title(stem)
    ax[1].imshow(pm, cmap='magma'); ax[1].set_title('filament probability')
    inst = np.zeros(img.shape, np.int32)
    for k, m in enumerate(masks, 1): inst[m] = k
    ax[2].imshow(img, cmap='gray'); ax[2].imshow(np.ma.masked_equal(inst, 0), cmap='tab20', alpha=0.6)
    ax[2].contour(sm > 128, [0.5], colors='cyan', linewidths=0.5); ax[2].set_title(f'{len(masks)} filaments (cyan: spines)')
    for a in ax: a.axis('off')
    plt.show()